# **Langgraph Agents Evaluation - Websearch Tools, Token usages, Information Delivery**

### Rafael Galán López

## **Introduccion y Marco de Evaluacion**

El presente notebook (`notebooks/agent_evaluations/langgraph_agents_evaluation.ipynb`) tiene como objetivo realizar un analisis tecnico y comparativo sobre el rendimiento de los agentes basados en LangGraph desarrollados en el modulo `src/rafael_langgraph_websearchagent/agents/`.

---

### 1. Foco de Evaluacion Principal: `A002_langgraph_agent_refined.py`

En la primera parte de este estudio nos centramos en evaluar a fondo el agente Refined (`A002`), el cual representa la base de los modelos Refined Plus `A003` y Refined Plus v2 `A004`. Analizaremos el comportamiento de su herramienta de busqueda web (`TavilySearch`) bajo los siguientes ejes clave:

1. **Estructura y Payload de la `search_tool`:** Analisis de la informacion devuelta por Tavily, la API inyectada en el prompt inicial mediante `bind_tools()` y el coste en tokens del `ToolMessage`.
2. **Generacion de Snippets y Score:** Evaluacion del metodo de extraccion de texto, la precision del score de similitud semantica y la relevancia real de los fragmentos recuperados frente a la consulta del usuario.
3. **Fidelidad Informativa vs. Alucinacion:** Diagnostico sobre como el LLM procesa las observaciones devueltas, identificando riesgos de sintesis incompleta o relleno con conocimiento estatico.

---

### 2. Evolucion y Hoja de Ruta de Optimizacion

Tras diagnosticar las limitaciones de la version Refined (`A002`), el notebook aborda la transicion hacia las versiones mejoradas:

- **Agent Refined Plus v1** (`A003_langgraph_agent_refined_plus.py`): Evaluacion de la desinfeccion de payload, truncado de respuesta y el filtro semantico estricto con Pydantic (`relevance_grader`).
- **Agent Refined Plus v2** (`A004_langgraph_agent_refined_plusv2.py`): Analisis de la version final optimizada, demostrando la flexibilizacion del evaluador de relevancia y la aplicacion de un limite de iteraciones para maximizar la precision reduciendo drasticamente el consumo de tokens.

## **Configuracion del entorno e Importacion de modulos**

En esta celda inicial se cargan las dependencias del proyecto, las variables de entorno necesarias para autenticar las peticiones a los modelos de lenguaje (LLM) y al motor de busqueda web, asi como las herramientas y agentes desarrollados en el paquete `src.rafael_langgraph_websearchagent`.

#### Requisitos de API Keys en `.env`
Para la correcta ejecucion de las celdas de evaluacion, es necesario contar con un archivo `.env` en la raiz del proyecto que contenga las siguientes claves de API[cite: 7]:

* `ANTHROPIC_API_KEY`: Clave de acceso al modelo de lenguaje (Claude de Anthropic).
* `TAVILY_API_KEY`: Clave de acceso al motor de busqueda y extraccion web (Tavily).

In [ ]:
import sys
from dotenv import load_dotenv

# Añadir la raíz del proyecto al sys.path para importar correctamente el paquete local
sys.path.append("../../")

# Cargar API Keys de Anthropic y Tavily
load_dotenv()

# Importaciones de LangChain y LangGraph
from langchain.chat_models import init_chat_model
from langchain_tavily import TavilySearch, TavilyExtract
from langchain.messages import HumanMessage, AIMessage, ToolMessage, SystemMessage

# Importación de los agentes locales desde el paquete
from src.rafael_langgraph_websearchagent.agents.A002_langgraph_agent_refined import agent as agent_refined
from src.rafael_langgraph_websearchagent.agents.A003_langgraph_agent_refined_plus import agent as agent_plus_v1
from src.rafael_langgraph_websearchagent.agents.A004_langgraph_agent_refined_plusv2 import agent as agent_plus_v2

print("✅ Entorno e importaciones cargadas correctamente.")

## **1. Explicación del Agente A002 Refined y de la search_tool**

En esta sección se detalla la arquitectura técnica y el mecanismo de funcionamiento de la herramienta de búsqueda web (`TavilySearch`) integrada en el agente Refined (`A002_langgraph_agent_refined.py`). Se describe la estructura del payload devuelto por la herramienta, el criterio de decisión del modelo para invocar la API externa y cómo se procesa e integra dicha información en el estado del grafo.

### **1.1 Flujo de ejecución de la búsqueda web en A002 Refined**

El proceso desde que el agente decide buscar en internet hasta que procesa la respuesta sigue tres etapas clave:

**1. Consulta y Formato de Respuesta**
El modelo formula una consulta a la API de Tavily. La herramienta devuelve una lista de diccionarios JSON (`list[dict]`) donde cada elemento contiene atributos estándar: `title`, `url`, `content` (snippet de texto) y `score` (puntuación de relevancia).

**2. Integración en el Estado (ToolMessage)**
LangGraph recibe la respuesta JSON, la convierte en un texto plano (string serializado) y la inyecta automáticamente en el historial de la conversación empaquetada como un objeto `ToolMessage`. Este mensaje asocia el contenido extraído con el identificador único de la petición (`tool_call_id`).

**3. Impacto en el Consumo de Tokens**
En la versión `A002_refined` no existe ningún filtro por relevancia ni truncado de texto. Todo el payload se vuelca íntegro en el estado, lo que inyecta miles de caracteres de golpe y dispara drásticamente el consumo de tokens de entrada (`prompt_tokens`) en las siguientes iteraciones del agente. Esto se solucionará en `A003_plus_v1` y `A004_plus_v2`

### **1.2 Criterio de Activación de la Herramienta de Búsqueda Web**

Los agentes de esta comparativa (`A002_refined`, `A003_plus_v1` y `A004_plus_v2`) comparten como motor base un modelo cuyo límite de conocimiento estático (*knowledge cutoff*) se sitúa en **julio de 2024**. Por tanto, la activación de la herramienta de búsqueda depende del equilibrio entre el conocimiento preentrenado del modelo y la necesidad de datos actualizados.

**Mecanismo de Decisión del Agente**
* **Uso del conocimiento estático:** Para consultas sobre hechos históricos anteriores a julio de 2024, conceptos teóricos, programación o razonamiento lógico, el agente responde directamente sin recurrir a la red.
* **Activación de Tavily Search:** Ante preguntas sobre eventos posteriores a julio de 2024, datos en tiempo real o información muy específica no garantizada en sus pesos internos, el agente emite un `AIMessage` invocando la herramienta de búsqueda.

**Optimización mediante Prompting**
Aunque los LLMs incorporan una heurística por defecto para determinar cuándo necesitan buscar información externa, es una buena práctica reforzar este criterio explícitamente en el *System Prompt*. De este modo se evita tanto el uso innecesario de la herramienta en consultas simples como la alucinación de datos en eventos recientes.

In [ ]:
# Prompt que no requiere búsqueda en internet
prompt = "¿Cual es el número atómico del Hierro?"
respuesta = agent_refined.invoke({"messages": [HumanMessage(content=prompt)]})

print("=== FLUJO DE CONVERSACIÓN ===")
for msg in respuesta["messages"]:
    msg.pretty_print()

In [ ]:
# Prompt que requiere búsqueda en internet
prompt = "¿Quién ganó los Premios Oscar 2025 en la categoría de Mejor Película y quién obtuvo el premio a Mejor Actor Principal?"
respuesta = agent_refined.invoke({"messages": [HumanMessage(content=prompt)]})

print("=== FLUJO DE CONVERSACIÓN ===")
for msg in respuesta["messages"]:
    msg.pretty_print()

### **1.3 Cálculo y Acumulación de Métricas de Tokens en A002 Refined**

El agente `A002_refined` monitoriza el consumo de recursos apoyándose en dos funciones auxiliares dentro del flujo de LangGraph para registrar el gasto en cada iteración.

**1. Consumo del LLM (`calculate_token_usage`)**
En cada ejecución del nodo `llm_call`, esta función extrae la propiedad `usage_metadata` que devuelve la API de Anthropic y acumula los valores en el estado:
* **Tokens de Prompt (`prompt_tokens`):** Lo que lee el modelo en cada turno (mensaje del sistema, historial y payloads de búsquedas anteriores).
* **Tokens de Respuesta (`completion_tokens`):** Lo que genera el modelo (la respuesta final o los argumentos para llamar a una herramienta).
* **Tokens Totales (`total_tokens`):** La suma acumulada de entrada y salida a lo largo de todas las llamadas al modelo (`llm_calls`).

**2. Estimación en la herramienta (`tool_node`)**
Como la API de Tavily Search no pasa por el modelo y no devuelve tokens oficiales, el nodo `tool_node` calcula una métrica propia llamada `tool_tokens`. Utiliza una fórmula de conversión heurística basada en la densidad media del texto (1 token $\approx$ 4 caracteres):

$$\text{tool\_tokens} = \max\left(1, \left\lfloor \frac{\text{len}(\text{content\_str})}{4} \right\rfloor\right)$$

Esto permite medir el volumen de texto bruto recuperado de internet antes de que sea inyectado en el historial y se convierta en `prompt_tokens` para el siguiente turno del modelo.

**3. Consolidación de métricas (`report_node` y `print_token_and_tool_report`)**
Al finalizar la ejecución, el nodo `report_node` reúne las métricas acumuladas en el estado y genera un mensaje final de tipo `AIMessage` con el resumen consolidado de uso de tokens y herramientas utilizadas.

In [ ]:
# OBSERVAR LAS MÉTRICAS DE TOKENS CON ESTE PROMPT QUE USA TAVILY SEARCH
prompt = "¿Cuáles son los titulares y noticias más destacados de la actualidad internacional en este momento?"
respuesta = agent_refined.invoke({"messages": [HumanMessage(content=prompt)]})

print("=== FLUJO DE CONVERSACIÓN ===")
for msg in respuesta["messages"]:
    msg.pretty_print()

## **2. Evaluación del agente A002 Refined y su search_tool**

En esta sección se evalúa el rendimiento práctico del agente `A002_refined` analizando tres aspectos críticos de su arquitectura: la **calidad de la respuesta generada tras la consulta web**, la **sobrecarga de tokens inyectados al estado** y la **latencia del flujo**.

In [ ]:
import time
import json

# Medición de latencia y ejecución sobre consulta compleja / actual
prompt_eval_detallada = "¿Cuáles son los principales avances en regulación de Inteligencia Artificial en la Unión Europea y Estados Unidos en 2025?"

inicio = time.perf_counter()
estado_eval = agent_refined.invoke({"messages": [HumanMessage(content=prompt_eval_detallada)]})
latencia_total = time.perf_counter() - inicio

# Extracción del ToolMessage para auditar el payload crudo
tool_messages = [m for m in estado_eval["messages"] if isinstance(m, ToolMessage)]

print(f"Tiempo total de ejecución (Latencia): {latencia_total:.2f} segundos")
print(f"Total de ToolMessages generados: {len(tool_messages)}")

if tool_messages:
    payload_crudo = tool_messages[0].content
    print(f"Longitud total del string inyectado al estado: {len(payload_crudo):,} caracteres")
    print("\n--- MUESTRA DE LOS PRIMEROS 800 CARACTERES DEL PAYLOAD ---")
    print(payload_crudo[:800] + "\n...")

print("\n--- RESPUESTA GENERADA POR EL AGENTE ---")
print(estado_eval["messages"][-2].content)

print(estado_eval["messages"][-1].content)

### **2.1 Auditoría del Payload, Snippets y Scores de `TavilySearch`**

Al inspeccionar el objeto `ToolMessage` generado durante la ejecución de `A002_refined`, se observan tres comportamientos estructurales en la recuperación de información:

**1. Sobrecarga de Metadatos y Ausencia de Sanitización**
La configuración por defecto (`TavilySearch(max_results=5)`) serializa la respuesta completa de la API en una cadena de texto plano que incluye URLs extensas, metadatos internos y fragmentos completos sin limpiar. Todo este bloque se inyecta de forma íntegra en `MessagesState`, lo que provoca que la segunda llamada al LLM (`llm_call`) deba procesar miles de `prompt_tokens` adicionales.

**2. Calidad de los Snippets y Ruido Semántico**
Tavily extrae fragmentos de texto (`content`) basados en coincidencia de palabras clave y relevancia semántica. Sin embargo, al no existir un umbral mínimo de filtrado, el payload suele arrastrar resultados con bajo nivel informativo (menús de navegación, encabezados de prensa o snippets truncados a mitad de frase) que ocupan espacio en la ventana de contexto sin aportar valor a la respuesta.

**3. Ignorancia del Score de Relevancia**
Cada diccionario devuelto por Tavily incluye un campo numérico `score` (entre `0` y `1`) que indica la confianza semántica del resultado frente a la consulta. En `A002_refined`, este indicador se ignora por completo: un resultado con un `score` de `0.35` recibe exactamente el mismo peso y espacio en el prompt que uno con `0.92`.

In [ ]:
# Prueba de estrés: Consulta sobre un evento ficticio o altamente específico para evaluar manejo de incertidumbre
prompt_fidelidad = "¿Qué acuerdos exactos se firmaron en la Cumbre Internacional de Computación Cuántica de Madrid en septiembre de 2026?"

estado_fidelidad = agent_refined.invoke({"messages": [HumanMessage(content=prompt_fidelidad)]})

print("--- DIAGNÓSTICO DE FIDELIDAD ---")
print(estado_fidelidad["messages"][-2].content)
print(estado_fidelidad["messages"][-1].content)

### **2.2 Diagnóstico de Fidelidad y Limitaciones Críticas de `A002_refined`**

A partir de las pruebas de rendimiento, consumo y fidelidad realizadas sobre `A002_refined`, se extraen las siguientes conclusiones técnicas que justifican la necesidad de evolucionar la arquitectura:

* **Ineficiencia en la relación Coste-Información:** El agente invierte una cantidad desproporcionada de `prompt_tokens` en leer texto irrelevante o repetitivo procedente de los 5 resultados brutos de Tavily. Esto incrementa tanto el coste económico por ejecución como la latencia de respuesta del modelo.
* **Riesgo de Contaminación de Contexto (Context Stuffing):** Al introducir resultados con bajo `score` semántico sin validación previa, se obliga al LLM a discriminar el ruido por sí mismo. En consultas ambiguas o con resultados escasos, la presencia de fragmentos parcialmente relacionados aumenta la probabilidad de que el modelo combine hechos de distintas fuentes de forma errónea o rellene lagunas con su conocimiento estático.
* **Ausencia de Control de Iteraciones y Reintentos Inteligentes:** Si la primera búsqueda devuelve resultados pobres (`score` bajo), `A002_refined` carece de un mecanismo evaluador que descarte la observación y reformule la consulta; simplemente genera una respuesta con los datos defectuosos disponibles.

Estas tres deficiencias constituyen el punto de partida para el diseño de **`A003_langgraph_agent_refined_plus.py`**, donde se introducirá limpieza de payload, truncado controlado y un nodo evaluador de relevancia estructurado con Pydantic.

## **3. Arquitectura y Evaluación de Agent Refined Plus v1 (Primeras mejoras)**

En esta sección se analiza la primera iteración evolutiva del agente (`A003_plus_v1`). A diferencia de `A002_refined`, donde la información recuperada de internet se volcaba directamente en el historial sin ningún tipo de procesamiento previo, esta versión introduce una capa intermedia de control de calidad y optimización de contexto antes de permitir que el LLM redacte la respuesta final.

### **3.1 Novedades Arquitectónicas Respecto a `A002_refined`**

El diseño de `A003_plus_v1` ataca directamente la sobrecarga de tokens y la contaminación de contexto mediante tres mecanismos principales:

**1. Sanitización y Desinfección del Payload**
En lugar de serializar el diccionario completo de Tavily en texto bruto, el nodo de procesamiento limpia la respuesta eliminando metadatos superfluos (scores internos, estructuras JSON anidadas o campos innecesarios) y conserva únicamente los atributos esenciales para el razonamiento: título, fuente y contenido textual.

**2. Truncamiento Controlado de Contenido**
Para evitar que artículos extensos disparen el consumo de `prompt_tokens` en los turnos posteriores, se aplica un límite máximo de caracteres por fragmento recuperado. Con ello se garantiza un techo predecible en el tamaño del `ToolMessage` inyectado al estado.

**3. Filtro Semántico Estricto con Pydantic (`relevance_grader`)**
Se incorpora un nodo evaluador que utiliza salidas estructuradas (*Structured Outputs* mediante esquemas de Pydantic) para auditar si los resultados obtenidos realmente responden a la intención de la consulta del usuario:
* Si los fragmentos superan el criterio de relevancia, el flujo avanza hacia la generación de la respuesta final.
* Si se consideran insuficientes o irrelevantes, el grafo descarta la observación y fuerza al agente a reformular la búsqueda en una nueva iteración.

In [ ]:
import time

# Consulta estándar para comparar directamente la reducción de tokens frente a A002_refined
prompt_eval_v1 = "¿Cuáles son los principales avances en regulación de Inteligencia Artificial en la Unión Europea y Estados Unidos en 2025?"

inicio = time.perf_counter()
estado_v1 = agent_plus_v1.invoke({"messages": [HumanMessage(content=prompt_eval_v1)]})
latencia_v1 = time.perf_counter() - inicio

tool_messages_v1 = [m for m in estado_v1["messages"] if isinstance(m, ToolMessage)]

print(f"Tiempo total de ejecución (Latencia): {latencia_v1:.2f} segundos")
print(f"Búsquedas ejecutadas (ToolMessages): {len(tool_messages_v1)}")

if tool_messages_v1:
    payload_limpio = tool_messages_v1[-1].content
    print(f"Longitud del payload sanitizado/truncado: {len(payload_limpio):,} caracteres")

print("\n--- RESPUESTA GENERADA POR A003 PLUS V1 ---")
print(estado_v1["messages"][-2].content)
print(estado_v1["messages"][-1].content)

In [ ]:
# Consulta compleja o multifacética para observar el comportamiento de reintentos del filtro estricto
prompt_estres_v1 = "¿Qué impacto ha tenido la última revisión de tipos de interés del BCE en las hipotecas variables en España durante este trimestre?"

inicio_estres = time.perf_counter()
estado_estres_v1 = agent_plus_v1.invoke({"messages": [HumanMessage(content=prompt_estres_v1)]})
latencia_estres = time.perf_counter() - inicio_estres

print(f"Latencia bajo consulta compleja: {latencia_estres:.2f} segundos")
print("--- REPORTE DE ITERACIONES Y TOKENS (PRUEBA DE ESTRÉS) ---")
print(estado_estres_v1["messages"][-1].content)

### **3.2 Hallazgos y Nueva Limitación Detectada en `A003_plus_v1`**

La evaluación comparativa frente a `A002_refined` confirma una mejora sustancial en la limpieza del estado, pero revela un nuevo cuello de botella operativo:

* **Reducción drástica de `tool_tokens` por llamada:** La desinfección del payload y el truncamiento reducen significativamente el volumen de caracteres inyectados en cada `ToolMessage`, eliminando el ruido estructural de la API de Tavily.
* **Mayor precisión en la respuesta final:** Al filtrar fragmentos con bajo aporte informativo antes de la síntesis, el LLM trabaja sobre un contexto más limpio y reduce el riesgo de mezclar hechos no relacionados.
* **Rigidez excesiva y riesgo de bucles (`relevance_grader`):** Al aplicar un criterio booleano o excesivamente estricto en el evaluador de Pydantic, el agente tiende a rechazar resultados válidos cuando la consulta es compleja o requiere combinar varias fuentes parciales. Esto provoca ciclos repetitivos de búsqueda (`llm_call` $\rightarrow$ `tool_node` $\rightarrow$ `grader`) que disparan nuevamente el total de llamadas al modelo y la latencia global.

Esta limitación motiva el diseño final en **`A004_langgraph_agent_refined_plusv2.py`**, donde se flexibilizará el criterio de evaluación y se impondrá un límite explícito de iteraciones.

## **4. Arquitectura y Evaluación de Agent Refined Plus v2**

En esta sección se analiza la versión final optimizada del agente (`A004_plus_v2`). Su diseño parte directamente de las lecciones extraídas en `A003_plus_v1`: conserva las ventajas de limpieza de payload y truncamiento controlado, pero corrige el cuello de botella provocado por la rigidez del evaluador semántico y los bucles de reintento.

### **4.1 Mejoras Estructurales y Correcciones Respecto a `A003_plus_v1`**

El paso a `A004_plus_v2` introduce dos ajustes arquitectónicos clave que estabilizan el comportamiento del grafo y maximizan su eficiencia operativa:

**1. Flexibilización del Evaluador de Relevancia (`relevance_grader`)**
En `A003_plus_v1`, el criterio booleano estricto descartaba conjuntos de resultados cuando solo aportaban información parcial. En `A004_plus_v2`, el esquema de evaluación y su prompt asociado se calibran bajo un enfoque pragmático: se valida la observación siempre que contenga datos útiles y verificables para construir una respuesta fundada, evitando rechazos innecesarios ante consultas complejas o multifacéticas.

**2. Límite Explícito de Iteraciones (Control de Bucle)**
Para garantizar que el agente nunca quede atrapado en ciclos infinitos de búsqueda y evaluación cuando la información no existe en la red o es ambigua, se incorpora un contador de reintentos en el estado del grafo. Al alcanzar el umbral máximo permitido, el flujo fuerza la salida hacia la síntesis final, permitiendo al modelo responder con la mejor evidencia disponible o declarar explícitamente la falta de datos sin desperdiciar llamadas adicionales al LLM.

In [ ]:
import time

# Se emplea la misma consulta compleja que estresó a A003_plus_v1 para contrastar la mejora
prompt_eval_v2 = "¿Qué impacto ha tenido la última revisión de tipos de interés del BCE en las hipotecas variables en España durante este trimestre?"

inicio = time.perf_counter()
estado_v2 = agent_plus_v2.invoke({"messages": [HumanMessage(content=prompt_eval_v2)]})
latencia_v2 = time.perf_counter() - inicio

tool_messages_v2 = [m for m in estado_v2["messages"] if isinstance(m, ToolMessage)]

print(f"Tiempo total de ejecución (Latencia): {latencia_v2:.2f} segundos")
print(f"Búsquedas ejecutadas (ToolMessages): {len(tool_messages_v2)}")

print("\n--- RESPUESTA GENERADA POR A004 PLUS V2 ---")
print(estado_v2["messages"][-2].content)
print(estado_v2["messages"][-1].content)

In [ ]:
# Prueba con evento inexistente para verificar que el límite de iteraciones corta el bucle de forma limpia
prompt_limite_v2 = "¿Qué acuerdos exactos se firmaron en la Cumbre Internacional de Computación Cuántica de Madrid en septiembre de 2026?"

inicio_lim = time.perf_counter()
estado_limite_v2 = agent_plus_v2.invoke({"messages": [HumanMessage(content=prompt_limite_v2)]})
latencia_lim = time.perf_counter() - inicio_lim

print(f"Latencia con control de iteraciones activo: {latencia_lim:.2f} segundos")
print("\n--- RESPUESTA ANTE AUSENCIA DE DATOS ---")
print(estado_limite_v2["messages"][-2].content)
print(estado_limite_v2["messages"][-1].content)

### **4.2 Balance de Fortalezas y Ventajas de `A004_plus_v2`**

La evaluación de `A004_plus_v2` confirma que esta arquitectura logra el equilibrio óptimo entre coste computacional, latencia y calidad informativa frente a sus predecesores:

* **Eficiencia Sostenida en el Consumo de Tokens:** Mantiene la reducción drástica de `prompt_tokens` y `tool_tokens` lograda mediante la sanitización y el truncado del payload, pero sin incurrir en el gasto extra derivado de múltiples reintentos fallidos.
* **Latencia Predecible y Acotada:** Al combinar un evaluador semántico flexible con un tope estricto de iteraciones, los tiempos de respuesta se estabilizan incluso ante consultas complejas o escenarios donde la web no ofrece resultados concluyentes.
* **Alta Fidelidad y Manejo Seguro de la Incertidumbre:** El agente aprovecha información parcial relevante para sintetizar respuestas completas y bien fundamentadas, al tiempo que conserva la capacidad de reconocer la ausencia de datos sin caer en alucinaciones cuando se agota el límite de búsquedas.

## **5. Conclusiones Generales**

El recorrido experimental realizado a lo largo de las tres arquitecturas demuestra que el rendimiento de un agente de búsqueda web en LangGraph no depende únicamente de la capacidad del modelo de lenguaje, sino del diseño de la tubería que gobierna la entrada y validación de los datos externos. Mientras que la versión base (`A002_refined`) penaliza el coste y la limpieza del contexto al volcar sin filtro todos los resultados de Tavily, la primera evolución (`A003_plus_v1`) logra reducir drásticamente el ruido mediante sanitización y truncamiento, aunque su evaluador semántico estricto introduce el riesgo de bucles y latencias variables ante consultas complejas.

La arquitectura final (`A004_plus_v2`) consolida el equilibrio óptimo entre eficiencia operativa y calidad informativa al calibrar el evaluador de relevancia bajo un criterio pragmático e imponer un límite explícito de iteraciones en el estado. De este modo, el agente mantiene un consumo de tokens bajo y predecible, sintetiza respuestas fieles apoyándose en evidencias parciales cuando es necesario y garantiza tiempos de ejecución acotados incluso frente a consultas donde no existe información disponible en la red.